# F1-Safe Node Swap — Cosine Similarity Optimisation

Computes per-node cosine similarity between GT and predicted node labels under the
best-of-5-orders MCS phi, then applies a brute-force F1-preserving swap to maximise
cosine similarity without changing the matched edge count.

**Algorithm:**
1. Reconstruct best phi (gt_node_id → pred_node_id) via best-of-5-orders MCS.
2. For every pair (i, j) of GT nodes in phi: if swapping phi[i] ↔ phi[j] improves
   mean cosine AND leaves the matched edge count unchanged, accept the swap.
3. Also try reassigning any GT node to an unassigned pred node (Case 2) and
   assigning unassigned GT nodes to unassigned pred nodes (Case 3).
4. Iterate until convergence (no further valid swap found).
5. Embed all labels with `text-embedding-3-small` and compute cosine similarity.

**Inputs:**
- GT JSON: `data/raw/project-1-at-2026-05-13-11-03-356a5d2b.json`
- Pred CSV: `data/output/v10/pipeline_v10_combined_210_step5_1_v2.csv`

**Output:**
- `data/output/v10/error_analysis_gt0513/matched_node_cosine_fullphi_f1safe.csv`

In [ ]:
import json
import os
import re
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from openai import OpenAI

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('.').resolve()))

import graph_eval_functions as gef

print('Imports OK')


In [ ]:
# ---- Config ----
ROOT     = Path('..').resolve() if Path('..').joinpath('data').exists() else Path('.').resolve()
GT_JSON  = ROOT / 'data/raw/EmpiriGraph-Psy_gold_annotation.json'
CSV      = ROOT / 'data/output/pipeline_gpt54_gpt52_210/results_step5_1.csv'
OUT_DIR  = ROOT / 'data/output/node_validation'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV  = OUT_DIR / 'matched_node_cosine_fullphi_f1safe.csv'

ORDERS      = ['degree_desc', 'degree_asc', 'label', 'random_11', 'random_29']
TIMEOUT     = 30.0
EMBED_MODEL = 'text-embedding-3-small'

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print(f'GT_JSON : {GT_JSON}')
print(f'CSV     : {CSV}')
print(f'OUT_CSV : {OUT_CSV}')


In [ ]:
# ---- Load data ----
with open(GT_JSON) as f:
    gt_articles = json.load(f)
gt_by_id = {int(a['id']): a for a in gt_articles}

df_csv = pd.read_csv(CSV, dtype=str)
df_csv['num_id'] = df_csv['article_id'].str.replace('json_', '').astype(int)
overlap = sorted(set(df_csv['num_id']) & set(gt_by_id.keys()))

print(f'GT articles : {len(gt_by_id)}')
print(f'Pred rows   : {len(df_csv)}')
print(f'Overlap     : {len(overlap)}')


In [ ]:
# ---- Helper functions ----

def get_label(G, n):
    return str(G.nodes[n].get('label', n)).strip()


def compute_matched(G_gt, G_pred, phi, pred_edge_set, pred_corr):
    """Count matched edges given phi (gt_node_id -> pred_node_id)."""
    return sum(
        1 for gu, gv, d in G_gt.edges(data=True)
        if phi.get(gu) and phi.get(gv) and
        (phi[gu], phi[gv], d['edge_type']) in
        (pred_corr if d['edge_type'] == 'correlational' else pred_edge_set)
    )


def cosine(a, b):
    a = a / (np.linalg.norm(a) + 1e-12)
    b = b / (np.linalg.norm(b) + 1e-12)
    return float(np.dot(a, b))


print('Helpers defined.')


In [ ]:
# ---- Step 1: Reconstruct best phi and collect all node labels ----
# Runs MCS for each article to get phi, collects all GT/pred label pairs.
# This cell builds the full_phi_data list used in subsequent cells.

full_phi_data = []  # list of dicts: aid, G_gt, G_pred, phi, gt_id2lbl, pred_id2lbl

for i, aid in enumerate(overlap):
    row_df = df_csv[df_csv['num_id'] == aid]
    if row_df.empty:
        continue
    G_gt   = gef.preprocess_graph_for_eval(gef.parse_gt_graph(gt_by_id[aid]))
    G_pred = gef.preprocess_graph_for_eval(gef.parse_pred_graph(row_df.iloc[0], 'step5_1'))
    if G_gt.number_of_edges() == 0:
        continue

    # Best-of-5 MCS
    best_res, best_phi = None, {}
    for s in ORDERS:
        res = gef.mcs_f1_with_order(G_gt, G_pred, TIMEOUT, s)
        phi = res.pop('best_phi', {})
        if best_res is None or res['edge_f1'] > best_res.get('edge_f1', 0):
            best_res, best_phi = res, phi

    full_phi_data.append({
        'aid':       aid,
        'G_gt':      G_gt,
        'G_pred':    G_pred,
        'phi':       best_phi,
        'gt_id2lbl':  {n: get_label(G_gt,   n) for n in G_gt.nodes()},
        'pred_id2lbl':{n: get_label(G_pred, n) for n in G_pred.nodes()},
    })

    if (i + 1) % 40 == 0:
        print(f'  {i+1}/{len(overlap)} articles processed')

print(f'\nBuilt phi for {len(full_phi_data)} articles.')


In [ ]:
# ---- Step 2: Embed all unique labels ----

all_labels = set()
for entry in full_phi_data:
    all_labels.update(entry['gt_id2lbl'].values())
    all_labels.update(entry['pred_id2lbl'].values())
all_labels = list(all_labels)
print(f'Embedding {len(all_labels)} unique labels...')

all_embs = []
for i in range(0, len(all_labels), 2048):
    resp = client.embeddings.create(model=EMBED_MODEL, input=all_labels[i:i+2048])
    all_embs.extend([d.embedding for d in resp.data])

emb_map = {t: np.array(e, dtype=np.float32) for t, e in zip(all_labels, all_embs)}

# Pre-normalise
for k in emb_map:
    v = emb_map[k]
    emb_map[k] = v / (np.linalg.norm(v) + 1e-12)

def get_emb(label):
    return emb_map.get(str(label), np.zeros(1536, dtype=np.float32))

def cos_fast(la, lb):
    return float(np.dot(get_emb(la), get_emb(lb)))

print('Embeddings ready.')


In [ ]:
# ---- Step 3: F1-safe swap to convergence ----
# For each article, iteratively swap phi assignments if cosine improves
# AND the matched edge count is unchanged.
# Three cases:
#   Case 1: swap two assigned GT nodes
#   Case 2: reassign assigned GT node to unassigned pred node
#   Case 3: assign unassigned GT node to best unassigned pred node

total_candidates = 0
total_accepted   = 0
results = []

for entry in full_phi_data:
    aid          = entry['aid']
    G_gt         = entry['G_gt']
    G_pred       = entry['G_pred']
    phi          = dict(entry['phi'])          # gt_node_id -> pred_node_id
    gt_id2lbl    = entry['gt_id2lbl']
    pred_id2lbl  = entry['pred_id2lbl']
    all_gt_nodes = list(G_gt.nodes())
    all_pred_nodes = list(G_pred.nodes())

    # Pre-compute edge sets
    pes  = {(u, v, d['edge_type']) for u, v, d in G_pred.edges(data=True)}
    pc   = pes | {(v, u, 'correlational') for u, v, et in pes if et == 'correlational'}

    # Extend phi to cover all GT nodes (None if unassigned)
    phi = {gn: phi.get(gn) for gn in all_gt_nodes}
    assigned_pred     = {v for v in phi.values() if v is not None}
    unassigned_pred   = set(all_pred_nodes) - assigned_pred

    orig_matched = compute_matched(G_gt, G_pred, phi, pes, pc)

    cos_cache = {
        gn: cos_fast(gt_id2lbl[gn], pred_id2lbl[phi[gn]]) if phi[gn] else 0.0
        for gn in all_gt_nodes
    }

    while True:
        swaps_this = 0

        # Case 1: swap two assigned GT nodes
        assigned_gt = [gn for gn in all_gt_nodes if phi[gn] is not None]
        n = len(assigned_gt)
        for i in range(n):
            gi = assigned_gt[i]
            for j in range(i + 1, n):
                gj = assigned_gt[j]
                swap_c = (cos_fast(gt_id2lbl[gi], pred_id2lbl[phi[gj]]) +
                          cos_fast(gt_id2lbl[gj], pred_id2lbl[phi[gi]]))
                if swap_c <= cos_cache[gi] + cos_cache[gj] + 1e-9:
                    continue
                total_candidates += 1
                phi[gi], phi[gj] = phi[gj], phi[gi]
                if compute_matched(G_gt, G_pred, phi, pes, pc) == orig_matched:
                    cos_cache[gi] = cos_fast(gt_id2lbl[gi], pred_id2lbl[phi[gi]])
                    cos_cache[gj] = cos_fast(gt_id2lbl[gj], pred_id2lbl[phi[gj]])
                    swaps_this += 1
                    total_accepted += 1
                else:
                    phi[gi], phi[gj] = phi[gj], phi[gi]

        # Case 2: reassign assigned GT node to unassigned pred node
        for gi in assigned_gt:
            pi_old = phi[gi]
            for pj in list(unassigned_pred):
                new_c = cos_fast(gt_id2lbl[gi], pred_id2lbl[pj])
                if new_c <= cos_cache[gi] + 1e-9:
                    continue
                total_candidates += 1
                phi[gi] = pj
                assigned_pred.add(pj);    assigned_pred.discard(pi_old)
                unassigned_pred.add(pi_old); unassigned_pred.discard(pj)
                if compute_matched(G_gt, G_pred, phi, pes, pc) == orig_matched:
                    cos_cache[gi] = new_c
                    pi_old = pj
                    swaps_this += 1
                    total_accepted += 1
                else:
                    phi[gi] = pi_old
                    assigned_pred.add(pi_old); assigned_pred.discard(pj)
                    unassigned_pred.add(pj);   unassigned_pred.discard(pi_old)

        # Case 3: assign unassigned GT node to best unassigned pred node
        unassigned_gt = [gn for gn in all_gt_nodes if phi[gn] is None]
        for gi in unassigned_gt:
            best_p, best_c = None, -1
            for pj in unassigned_pred:
                c = cos_fast(gt_id2lbl[gi], pred_id2lbl[pj])
                if c > best_c:
                    best_c, best_p = c, pj
            if best_p is None or best_c <= 0:
                continue
            total_candidates += 1
            phi[gi] = best_p
            if compute_matched(G_gt, G_pred, phi, pes, pc) == orig_matched:
                cos_cache[gi] = best_c
                assigned_pred.add(best_p)
                unassigned_pred.discard(best_p)
                swaps_this += 1
                total_accepted += 1
            else:
                phi[gi] = None

        if swaps_this == 0:
            break

    # Collect results
    for gn in all_gt_nodes:
        pn = phi[gn]
        if pn is None:
            continue
        gl = gt_id2lbl[gn]
        pl = pred_id2lbl[pn]
        results.append({
            'article_id':       aid,
            'gt_node':          gl,
            'pred_node':        pl,
            'cosine_similarity':cos_cache[gn],
            'exact_match':      int(gl.lower().strip() == pl.lower().strip()),
        })

print(f'Candidates tested : {total_candidates}')
print(f'Accepted (F1-safe): {total_accepted}')
print(f'Rejected          : {total_candidates - total_accepted}')


In [ ]:
# ---- Step 4: Save and summarise ----

out_df = pd.DataFrame(results)
out_df.to_csv(OUT_CSV, index=False)
print(f'Saved {len(out_df)} rows -> {OUT_CSV}')

cos = out_df['cosine_similarity']
print(f'\n=== NODE COSINE SIMILARITY (F1-safe swap, n={len(out_df)}, '
      f'{out_df["article_id"].nunique()} articles) ===')
print(f'  Mean   : {cos.mean():.4f}')
print(f'  Median : {cos.median():.4f}')
print(f'  SD     : {cos.std():.4f}')
print(f'  Min    : {cos.min():.4f}  Max: {cos.max():.4f}')
print(f'  Exact match: {out_df["exact_match"].sum()} ({out_df["exact_match"].mean()*100:.1f}%)')
print()
tiers = [
    ('>=0.9 near-exact', cos >= 0.9),
    ('0.7-0.9 high',     (cos >= 0.7) & (cos < 0.9)),
    ('0.5-0.7 moderate', (cos >= 0.5) & (cos < 0.7)),
    ('0.3-0.5 weak',     (cos >= 0.3) & (cos < 0.5)),
    ('<0.3 low',          cos < 0.3),
]
for lbl, mask in tiers:
    print(f'  {lbl:22s}: {mask.sum():4d}  ({mask.mean()*100:.1f}%)')

per_art = out_df.groupby('article_id')['cosine_similarity'].mean()
print(f'\nPer-article mean: {per_art.mean():.4f} ± {per_art.std():.4f}')
print(f'  >= 0.7: {(per_art>=0.7).sum()} ({(per_art>=0.7).mean()*100:.1f}%)')
print(f'  <  0.5: {(per_art<0.5).sum()} ({(per_art<0.5).mean()*100:.1f}%)')

out_df.head(10)
